In [2]:
from pathlib import Path
import os
import random
import shutil
import string
import yaml

from _configs.country_config import *
from _configs.files_config import *
from _configs.run_config import *

from _utils.utils import *

### Genrate input files

In [3]:
print("Validation path: ", os.path.abspath(validation_path))
print("Template path: ", os.path.abspath(template_path))

Validation path:  /work/tuv89272/Calibration_Pipeline_v7_NG-SE/validation_1_copy_new_treatment_seeking_zero_beta
Template path:  /work/tuv89272/Calibration_Pipeline_v7_NG-SE/calibration_template_files


In [5]:
# init_inferred_pop_raster_path = Path(data_path) / input_population_file
# init_observed_pop_raster_path = Path(data_path) / observed_initial_population_file

init_inferred_pop_raster, metadata = read_raster(initial_population_projected_raster_path)
# init_observed_pop_raster, metadata = read_raster(init_observed_pop_raster_path)

nodata = metadata["NODATA_value"]

total_inferred_initial_population = np.sum(init_inferred_pop_raster[init_inferred_pop_raster != nodata])
# total_observed_initial_population = np.sum(init_observed_pop_raster[init_observed_pop_raster != nodata])

# percent_diff_inferred_observed = ((total_inferred_initial_population - total_observed_initial_population) / total_observed_initial_population) * 100
# abs_percent_diff_inferred_observed = abs(percent_diff_inferred_observed)

print(f"Total inferred population in initial raster ({initial_year}) to be used in validation: {total_inferred_initial_population:,.0f}")
# print(f"Total observed population in initial raster ({initial_year}): {total_observed_initial_population:,.0f}")

# raise ValueError(f"ERROR: The difference between inferred and observed initial population is {percent_diff_inferred_observed:.2f}%. Double check population raster is as expected and proceed with caution.")
# info(f"Percent difference between inferred and observed initial population: {percent_diff_inferred_observed:.2f}%")
# if abs_percent_diff_inferred_observed > 5:
#     raise ValueError(f"ERROR: The difference between inferred and observed initial population is {percent_diff_inferred_observed:.2f}% which exceeds the 5% threshold.")
# elif abs_percent_diff_inferred_observed > 2.5:
#     error(f"ERROR: The difference between inferred and observed initial population is {percent_diff_inferred_observed:.2f}% which exceeds the 5% threshold.")
# elif abs_percent_diff_inferred_observed > 1:
#     warn(f"WARNING: The difference between inferred and observed initial population is {percent_diff_inferred_observed:.2f}% which exceeds the 1% threshold.")

Total inferred population in initial raster (2011) to be used in validation: 20,357,570


In [9]:
# copy bin folder from template to validation path
template_bin = os.path.join(template_path, "bin")
validation_bin = os.path.join(validation_path, "bin")

if os.path.exists(template_bin):
    if os.path.exists(validation_bin):
        ok(f"Bin folder already exists in validation path: {validation_bin}. Skipping copy.\n")
    else:
        shutil.copytree(template_bin, validation_bin, copy_function=shutil.copy2, symlinks=True)
        ok(f"Copied bin folder from {template_bin} to {validation_bin}\n")
else:
    error(f"Bin folder not found in template path: {template_bin}. Please ensure it is available.\n")

# Paths
TEMPLATE = "input_population_bins.yml"


# # pop_calib_input_path = Path(PRE_SIM_RUN_PATH_INPUTS_DIR) / "input_beta_0.0000_access_0.000_pop_200000.yml"
# pop_calib_input_path = Path("/home/tuv89272/work/Calibration_Pipeline_v6_NG-SE/_pre_sim_200k_pop_run/input/") / "input_beta_0.0000_access_0.000_pop_200000.yml"
# # pop_calib_input_data = yaml.safe_load(pop_calib_input_path)
# with pop_calib_input_path.open("r", encoding="utf-8") as f:
#     pop_calib_input_data = yaml.safe_load(f)

# # birth_rate = pop_calib_input_data.get("birth_rate", -1.0)
# birth_rate = pop_calib_input_data.get("population_demographic", {}).get("birth_rate", -1.0) # value is nested in v6 config


# if(birth_rate < 0):
#     raise ValueError("Birth rate not found in input file. Please ensure the to calibrate population birth rate before running calibration and validation.")


info(f"Using birth rate: {birth_rate}")
info(f"Using validation population scale: {validation_population_scale}")
birth_rate_str = f"{birth_rate:.4f}"

os.makedirs(VALIDATION_RUN_INPUTS_DIR, exist_ok=True)

# Read templates
template_full_path = os.path.abspath(os.path.join(template_path, TEMPLATE))
print(f"Reading input template from: {template_full_path}")
if not os.path.exists(TEMPLATE):
    raise FileNotFoundError(f"Input template not found: {template_full_path}")

with open(TEMPLATE, "r", encoding="utf-8") as f:
    input_template_text = f.read()    

out_text = input_template_text.replace(f"{country_code}_initialpopulation#POPULATION#", initial_population_projected_raster_path)
out_text = out_text.replace("#BETA#", "").replace("#ACCESS_RATE#", "")
out_text = out_text.replace("#BIRTH_RATE#", birth_rate_str)
out_text = out_text.replace("#CALIBRATION_YEAR#", f"{calibration_year}")
out_text = out_text.replace("#COUNTRY_CODE#", f"{country_code}")
out_text = out_text.replace("#INITIAL_YEAR#", f"{initial_year}")
out_text = out_text.replace("#INPUT_PATH#", "input")
out_text = out_text.replace("#POPULATION_SCALE#", f"{validation_population_scale}")
out_text = out_text.replace("initialpopulation#POPULATION#", f"initpopulation_{initial_year}_inferred_for_sim")
out_text = out_text.replace(f"{country_code}_seasonality_1_location", f"{country_code}_seasonality")
out_text = out_text.replace("district.asc", "district_seq1.asc")

out_text = out_text.replace("cell_level_reporting: false", "cell_level_reporting: true")

out_name = os.path.abspath(os.path.join(VALIDATION_RUN_INPUTS_DIR, f"input_validation.yml"))

with open(out_name, "w", encoding="utf-8") as outf:
    outf.write(out_text)
    print(f"Created input file: {out_name}\n")
    
seasonality_file = f"{template_path}/{country_code}_seasonality_1_pattern.csv"
if SEASONALLITY_MODE == "one":  # Remove _1_pattern from filename for output
    if os.path.exists(seasonality_file):
        dest = os.path.abspath(os.path.join(VALIDATION_RUN_INPUTS_DIR, seasonality_file.replace("_1_pattern", "").replace(template_path + "/", "")))
        with open(seasonality_file, "r", encoding="utf-8") as srcf:
            with open(dest, "w", encoding="utf-8") as destf:
                destf.write(srcf.read())
        ok(f"Copied {seasonality_file} to {dest}")
    else:
        warn(f"Warning: required raster not found: {seasonality_file}. Please ensure it is available in the current directory.")
else:    # Remove _multiple_patterns from filename for output   
    seasonality_file = f"{template_path}/{country_code}_seasonality_multiple_patterns.csv"
    if os.path.exists(seasonality_file):
        dest = os.path.abspath(os.path.join(VALIDATION_RUN_INPUTS_DIR, seasonality_file.replace("_multiple_patterns", "").replace(template_path + "/", "")))
        with open(seasonality_file, "r", encoding="utf-8") as srcf:
            with open(dest, "w", encoding="utf-8") as destf:
                destf.write(srcf.read())
        ok(f"\nCopied {seasonality_file} to {dest}")
    else:
        warn(f"Warning: required raster not found: {seasonality_file}. Please ensure it is available in the current directory.")

print("")
    
# Copy other raster files that are needed for the model but not generated here (e.g. population, districts, traveltime, etc.)
other_files = [
    f"{calibration_analysis_path}/{country_code}_beta.asc",
    initial_population_projected_raster_path, 
    districts_raster_sequential_path, 
    travel_time_raster_path,
    treatment_seeking_raster_path,
    ]
for o_file in other_files:
    if os.path.exists(o_file):
        # print(f"\nCopying {o_file} to {VALIDATION_RUN_INPUTS_DIR}")
        # dest = os.path.abspath(os.path.join(VALIDATION_RUN_INPUTS_DIR, o_file.replace(f"{data_path}/", "").replace(f"{calibration_analysis_path}/", "")))
        dest = os.path.abspath(os.path.join(VALIDATION_RUN_INPUTS_DIR, o_file.replace(f"{data_path}/", "").replace(f"{calibration_analysis_path}/", "").replace(f"{generated_data_path}/", "")))
        with open(o_file, "r", encoding="utf-8") as srcf:
            with open(dest, "w", encoding="utf-8") as destf:
                destf.write(srcf.read())
        print(f"Copied {o_file} to {dest}")
    else:
        warn(f"Warning: required file not found: {o_file}. Please ensure it is available in the current directory.") 

✓ Bin folder already exists in validation path: validation_1_copy_new_treatment_seeking_zero_beta/bin. Skipping copy.

→ Using birth rate: 0.0362
→ Using validation population scale: 0.001
Reading input template from: /work/tuv89272/Calibration_Pipeline_v7_NG-SE/calibration_template_files/input_population_bins.yml
Created input file: /work/tuv89272/Calibration_Pipeline_v7_NG-SE/validation_1_copy_new_treatment_seeking_zero_beta/input/input_validation.yml

✓ Copied calibration_template_files/ng-se_seasonality_1_pattern.csv to /work/tuv89272/Calibration_Pipeline_v7_NG-SE/validation_1_copy_new_treatment_seeking_zero_beta/input/ng-se_seasonality.csv

Copied calibration_1_1_population_scale_one_pattern_25_replicates/analysis/ng-se_beta.asc to /work/tuv89272/Calibration_Pipeline_v7_NG-SE/validation_1_copy_new_treatment_seeking_zero_beta/input/ng-se_beta.asc
Copied generated/ng-se_population_backwards_projected_2011.asc to /work/tuv89272/Calibration_Pipeline_v7_NG-SE/validation_1_copy_new_trea

### Generate cmds.txt

In [10]:
validation_path = Path(validation_path)
VALIDATION_RUN_INPUTS_DIR    = Path(VALIDATION_RUN_INPUTS_DIR)
output_path     = Path(validation_output_path)
log_path        = Path(log_path)
script_path     = Path(script_path)
validation_path.mkdir(exist_ok=True)
log_path.mkdir(exist_ok=True)
output_path.mkdir(exist_ok=True)
script_path.mkdir(exist_ok=True)

# # Clean script, output, and log directories
# for path in (script_path, output_path, log_path):
#     if path.exists():
#         for item in path.iterdir():
#             if item.is_file() or item.is_symlink():
#                 item.unlink()
#             elif item.is_dir():
#                 shutil.rmtree(item)
#         print(f"Cleaned directory: {path}")
#     else:
#         path.mkdir(parents=True, exist_ok=True)
#         print(f"Created directory: {path}")

# Just clean script path
if script_path.exists():
    for item in script_path.iterdir():
        if item.is_file() or item.is_symlink():
            item.unlink()
        elif item.is_dir():
            shutil.rmtree(item)
    print(f"Cleaned directory: {script_path}")

cmds = []
for rep in range(1, validation_replicates + 1):
    input_rel  = (VALIDATION_RUN_INPUTS_DIR / "input_validation.yml").relative_to(validation_path)
    output_rel = (output_path / "validation_").relative_to(validation_path)
    log_rel    = (log_path / f"validation_rep_{rep}.log").relative_to(validation_path)

    cmd = (
        f"./bin/MalaSim -i {input_rel} -r SQLiteMonthlyReporter -j {rep} "
        f"-o {output_rel} -v 1 > {log_rel} 2>&1"
    )
    cmds.append(cmd)

job_name = ''.join(random.choices(string.ascii_letters + string.digits, k=6))

# Write commands to script file
script_file = os.path.join(script_path, f"cmds_{job_name}.txt")
with open(script_file, "w", encoding="utf-8") as f:
    for cmd in cmds:
        f.write(cmd + "\n")
ok(f"\nWrote {len(cmds)} commands to {script_file}")


Cleaned directory: validation_1_copy_new_treatment_seeking_zero_beta/script
✓ 
Wrote 15 commands to validation_1_copy_new_treatment_seeking_zero_beta/script/cmds_8jaXNM.txt


In [ ]:
queue_name = "nd"
host_id = "05"

host_name = "" if queue_name == "max" else f":host=nd{host_id}"

jobs_template_path = f"_templates"

with open(f"{jobs_template_path}/job_template.template", "r", encoding="utf-8") as f:
    job_template_text = f.read()
    job_template_text = job_template_text.replace("#QUEUE_NAME#", queue_name).replace("#HOST_NAME#", host_name)
    job_template_text = job_template_text.replace("#JOB_NAME#", job_name)
    
with open(f"{script_path}/job_template_{job_name}.pbs", "w", encoding="utf-8") as f:
    f.write(job_template_text)
    
with open(f"{jobs_template_path}/submit_jobs.template", "r", encoding="utf-8") as f:
    submit_template_text = f.read()
    submit_template_text = submit_template_text.replace("#QUEUE_NAME#", queue_name).replace("#HOST_NAME#", host_name)
    submit_template_text = submit_template_text.replace("#JOB_NAME#", job_name)
    submit_template_text = submit_template_text.replace("cmds.txt", f"{job_name}_cmds.txt")
    submit_template_text = submit_template_text.replace("job_template.pbs", f"job_template_{job_name}.pbs")

with open(f"{script_path}/submit_jobs_{job_name}.pbs", "w", encoding="utf-8") as f:
    f.write(submit_template_text)
    
print(f"Prepared job script and submit script for job: {job_name}")
full_script_path = os.path.abspath(script_path)
print(f"Full script path: {full_script_path}")
print(f"Submit script: {full_script_path}/submit_jobs_{job_name}.pbs")

FileNotFoundError: [Errno 2] No such file or directory: 'calibration_template_files/job_template.template'

After preparing submit script, submit using `qsub script/submit_jobs_<######>.pbs` or `qsub script/submit_jobs_*`